# Stable Diffusion

**Goal**: Using Stable diffusion to create images from text descriptions. Starts by working with each of the components of Stable Diffusion individually. Finishes by using Stable Diffusion pipeline that joins all of the components together, providing a much more convenient interface.

**Objectives**

- Use the CLIP model to create text embeddings.
- Decode latent vectors into an image with a VAE.
- Apply a denoising diffusion model to randomly-generated noise.
- Use a scheduler to perform denoising over multiple steps.
- Generate images matching a text description with a Stable Diffusion pipeline.

In [11]:
import sys

import diffusers
import matplotlib.pyplot as plt 
import torch
import transformers
from IPython.display import display
from PIL import Image
from torchinfo import summary
from tqdm.notebook import tqdm

In [12]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("diffusers version:", diffusers.__version__)
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("PIL version:", Image.__version__)

Platform: win32
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
---
diffusers version: 0.33.1
transformers version: 4.51.3
torch version: 2.2.2+cpu
PIL version: 10.2.0


In [13]:
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Using {device} device with {dtype} data type.")

Using cpu device with torch.float32 data type.


**Creating Text Embeddings**

Stable Diffusion attempts to produce an image aligned to a text description. We start by working on that text description, by converting human-readable text into an embedding. This means that the text will be represented by a bunch of numbers, which the model will be able to ingest. 

Stable Diffusion uses text embeddings from the Constrastive Language-Image Pre-Training (CLIP) model. We need two pieces from this model; first is the tokenizer that splits up a string into tokens, a word, part of a word, or single character.

In [15]:
tokenizer = transformers.CLIPTokenizer.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="tokenizer",
    torch_dtype=dtype
)
print(tokenizer)

CLIPTokenizer(name_or_path='CompVis/stable-diffusion-v1-4', vocab_size=49408, model_max_length=77, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	49406: AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	49407: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)


In [16]:
type(tokenizer)

transformers.models.clip.tokenization_clip.CLIPTokenizer

Example usage:

In [17]:
text = "Hello, world!"
result = tokenizer(text)

print(type(result))
print(result)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
{'input_ids': [49406, 3306, 267, 1002, 256, 49407], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [18]:
result["input_ids"]

[49406, 3306, 267, 1002, 256, 49407]

In [19]:
result.input_ids

[49406, 3306, 267, 1002, 256, 49407]

Decode Tokens:

In [20]:
for token in result.input_ids:
    print(tokenizer.decode(token))

<|startoftext|>
hello
,
world
!
<|endoftext|>


**Image generation prompt**

In [21]:
prompt = "A red bird flies through a blue sky over a green tree."

Tokenization:

In [22]:
text_tokens = tokenizer(
    prompt,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)

print(text_tokens.input_ids)
print(text_tokens.input_ids.shape)

tensor([[49406,   320,   736,  3329,  8070,  1417,   320,  1746,  2390,   962,
           320,  1901,  2677,   269, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407]])
torch.Size([1, 77])


Tokens for the empty string input:

In [23]:
uncond_tokens = tokenizer(
    "",
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print(uncond_tokens.input_ids)
print(uncond_tokens.input_ids.shape)

tensor([[49406, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407]])
torch.Size([1, 77])


These tokens will be converted into a text embedding with the text_encoder component.

In [24]:
embedder = transformers.CLIPTextModel.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="text_encoder",
    torch_dtype=dtype
)
embedder.to(device)  

# Print out a summary of this neural network
summary(embedder)

config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

Layer (type:depth-idx)                                       Param #
CLIPTextModel                                                --
├─CLIPTextTransformer: 1-1                                   --
│    └─CLIPTextEmbeddings: 2-1                               --
│    │    └─Embedding: 3-1                                   37,945,344
│    │    └─Embedding: 3-2                                   59,136
│    └─CLIPEncoder: 2-2                                      --
│    │    └─ModuleList: 3-3                                  85,054,464
│    └─LayerNorm: 2-3                                        1,536
Total params: 123,060,480
Trainable params: 123,060,480
Non-trainable params: 0

We call it with the token IDs to generate the embedding.

In [25]:
with torch.no_grad():  # No need for gradient calculations
    text_embedding = embedder(text_tokens.input_ids.to(device))

print(type(text_embedding))
print(text_embedding.keys())

<class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
odict_keys(['last_hidden_state', 'pooler_output'])


The  class and shape of the last_hidden_state attribute.

In [26]:
print("Class:", type(text_embedding.last_hidden_state))
print("Shape:",text_embedding.last_hidden_state.shape )

Class: <class 'torch.Tensor'>
Shape: torch.Size([1, 77, 768])


We call it with the token IDs to generate the embedding.

In [27]:
with torch.no_grad():
    uncond_embedding = embedder(uncond_tokens.input_ids.to(device))

print(uncond_embedding.last_hidden_state.shape)

torch.Size([1, 77, 768])


Concatenate the unconditioned embedding followed by the text embedding into a tensor called all_embeddings

In [29]:
all_embeddings = torch.cat(
    [
        uncond_embedding.last_hidden_state,
        text_embedding.last_hidden_state,
    ]
)

print(all_embeddings.shape)

torch.Size([2, 77, 768])


**Generating Random Latent Vectors**

Stable Diffusion uses a Variational Auto-Encoder (VAE) to generate images from latent vectors.

In [30]:
vae = diffusers.AutoencoderKL.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="vae",
    torch_dtype=dtype
)
vae.to(device)  # Run it on the GPU

summary(vae)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Layer (type:depth-idx)                             Param #
AutoencoderKL                                      --
├─Encoder: 1-1                                     --
│    └─Conv2d: 2-1                                 3,584
│    └─ModuleList: 2-2                             --
│    │    └─DownEncoderBlock2D: 3-1                738,944
│    │    └─DownEncoderBlock2D: 3-2                2,690,304
│    │    └─DownEncoderBlock2D: 3-3                10,754,560
│    │    └─DownEncoderBlock2D: 3-4                9,443,328
│    └─UNetMidBlock2D: 2-3                         --
│    │    └─ModuleList: 3-5                        1,051,648
│    │    └─ModuleList: 3-6                        9,443,328
│    └─GroupNorm: 2-4                              1,024
│    └─SiLU: 2-5                                   --
│    └─Conv2d: 2-6                                 36,872
├─Decoder: 1-2                                     --
│    └─Conv2d: 2-7                                 18,944
│    └─ModuleList: 2-8

This VAE is constructed mainly of convolutional layers, and it's able to produce outputs of various sizes

In [31]:
height = 512
width = 512

The encoder will downscale both the width and height by a factor of 8. The decoder will upscale by the same factor.

In [ ]:
scale_factor = 8

The number of channels in the latent vector is stored in the configuration:

In [ ]:
n_channels = vae.config.latent_channels
print(n_channels)

Latent vectors will be a 4-D tensor, representing (batch, channel, height, width). Don't be confused by the fact that there happen to be four channels. That is a complete coincidence.